# Analysis of shape evolution during growth of propagules

## 1. Documentation and Imports


Created on 22-03-2021

Author: Valentin Laplaud

This code aims to characterize the shape of propagules growing in microfluidic chips. It uses the binarized image of the propagules generated by 'ColorPpglSizeTrackingNB.ipynb'

In [1]:

# plotting stuff
import matplotlib as mpl
import matplotlib.pyplot as plt
from cycler import cycler

# numbers handling
import numpy as np
import pandas as pd

# signal processing 
from scipy.signal import savgol_filter, correlate, correlation_lags
from scipy.interpolate import interp1d

# images handling
from skimage import io
from skimage.filters import threshold_otsu, gaussian, laplace, sobel
from skimage.measure import label, regionprops, regionprops_table
from skimage.util import invert
from skimage.morphology import binary_opening, binary_closing, remove_small_holes
from skimage.color import rgb2gray
import cv2 as cv

# to hide known warnings
import warnings

# General system functions
import os
import shutil
import sys

import time

# my functions
sys.path.append('../')
import VallapFunc as vf

## 2. Define analysis functions

### 2.0 Subfunctions

2.0.1 Computing edge from binary image

In [2]:
def getEdge(BinImg):
    
    # Find largest contour 
    cnts, _ = cv.findContours(BinImg, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_NONE)
    cnt = max(cnts, key=cv.contourArea)

    # Create image with only largest object
    out = np.zeros(BinImg.shape, np.uint8)
    cv.drawContours(out, [cnt], -1, 255, cv.FILLED)
    BinImg = cv.bitwise_and(BinImg, out)

    # Detecting edges and computing their position
    edge = sobel(BinImg)>threshold_otsu(sobel(BinImg))
    indices = np.where(edge != [0])
    values = np.extract(edge != [0],edge)
    
    # Computing center of propagule using euclidian distance transform
    dist = cv.distanceTransform(BinImg, cv.DIST_L2, 3)
    regions = regionprops(BinImg,dist)
    center = regions[0].weighted_centroid
    
    # Edge coorinate relative to the center
    relativeIndicesX = indices[0]-center[0]
    relativeIndicesY = indices[1]-center[1]
    
    # Edge coordinate in circular system
    Alpha,Radius = vf.ToCirc(relativeIndicesX,relativeIndicesY, angle = 'deg')

    # Sorting coordinate based on angle
    FullCoordinates = np.stack((Alpha,Radius,relativeIndicesX,relativeIndicesY,values),axis=1)
    SortedCoord = FullCoordinates[FullCoordinates[:,0].argsort(),]
    
    SortedAlpha = SortedCoord[:,0]
    SortedRadius = SortedCoord[:,1]
    SortedEdgeX = SortedCoord[:,2]
    SortedEdgeY = SortedCoord[:,3]   
    SortedValues = SortedCoord[:,4]  
    
    return(SortedAlpha,SortedRadius,SortedEdgeX,SortedEdgeY,SortedValues)

2.0.2 Aligning and average a list of contours

In [ ]:
def MeanContour(StackList,datafilename,P,figname):    
      
    fig1 = plt.figure(dpi = 250)
    
    # Aligning all first image contours
    ContourData = pd.read_csv(P + '\\' + datafilename, index_col = 'Ind')
    ContourData['alignR'] = np.nan

    n =  int(ContourData['Img'].max())
    
    Alpha = ContourData[ContourData['Img'] == 0][['Alpha']].loc[StackList[0]].to_numpy()
    
    for i in range(n):
    
        fig1 = plt.figure(dpi=250)
        
        refR = ContourData[ContourData['Img'] == i][['R']].loc[StackList[0]].to_numpy()
        sig = np.concatenate((refR,refR)) - refR.mean()

        for s in StackList:
            currentR = ContourData[ContourData['Img'] == i][['R']].loc[s].to_numpy()

            sig2 = np.concatenate((currentR,currentR)) - currentR.mean()

            # Aligining with reference contour using cross corelation 
            corr = correlate(sig2, sig)

            lags = correlation_lags(len(sig), len(sig2))

            corr /= np.max(corr)

            lag = lags[np.argmax(corr)]

            alignR = np.roll(currentR, -lag)
            
            #plt.figure(fig1.number)
            #plt.plot(Alpha,alignR,'-', lw=0.5)

            ContourData.loc[(ContourData.index == s) & (ContourData['Img'] == i),'alignR'] = alignR[:,0]
        
        # Computing mean of aligned contours
        meanR = np.mean([ContourData.loc[(ContourData.index == s) & (ContourData['Img'] == i),'alignR'] for s in StackList],0)
        
        X,Y = vf.ToCart(Alpha,meanR, angle = 'deg')
        
        plt.figure(fig1.number)
        plt.plot(X,Y,'-', lw=0.5)
        
    fig1.savefig(P + '\\Figures\\MeanContour' + figname + '.png')
        
        
    return

### 2.1 Image analysis

Works on the binary image created by 'ColorPpglSizeTrackingNB'. First isolates the biggest binary object. Then computes the edges position and the center of the propagule using a euclidian distance transform and a weighted average based on intensity. Returns the positions of the edge relative to the center, in both cartesian and circular (angle in degree) coordinates.
    

In [4]:
def GetContours(StackList,P,datafilename, Scale, **kwargs):
    
    ContourData = pd.DataFrame(data=None,columns=['Time (min)','Img','X','Y','R','Alpha']) 

    
     #init and read kwargs
    ShapePlot = False
    
    for key, value in kwargs.items(): 
        if key == 'ShapePlot':
            ShapePlot = value
        else:
            print('Unknown key : ' + key + '. Kwarg ignored.')
    
    
    sizeData = pd.read_csv(P + '\\' + datafilename, index_col = 'Ind')
    
    for s in StackList:

        n = np.min([len(sizeData.loc[s]),49])
        jet_colorcycle = [plt.get_cmap('jet')(1. * i/n) for i in range(n)]
        mpl.rcParams['axes.prop_cycle'] = cycler(color=jet_colorcycle)

        if ShapePlot:
            
            fig1 = plt.figure(dpi = 250)
            plt.suptitle(s)
            plt.ion()
            plt.xlabel('(µm)')
            plt.ylabel('(µm)')
            plt.plot(0,0,'*r')                               
            plt.gca().set_aspect('equal', adjustable='box')
                
            fig2 = plt.figure(dpi = 250)
            plt.ion()
            plt.suptitle(s)
            plt.xlabel('Angle (°)')
            plt.ylabel('Radius (µm)')


        for i in range(n):
            
            if (sizeData.loc[s]['GoodFocus'].to_numpy()[i] == True):
                
                # time of image
                Ti = sizeData.loc[s]['Time (min)'].to_numpy()[i]
                
                # Loading binary image
                BinImg = io.imread(P + '\\Processed\\' + s + '_Binarized\\' + str(i) + '.tif')

                # Computing propagule edge
                Alpha,Radius,X,Y,Values = getEdge(BinImg)
                
                              
                # interpolation on regular grid
                f = interp1d(Alpha+180, Radius, fill_value='extrapolate')
                Alphanew = np.arange(0.5, 359.5, 0.2)
                Radiusnew = f(Alphanew)  
                
                # Smoothing
                SmoothRadius = savgol_filter(Radiusnew,7,3)  
                
      
                # Aligining with first image using cross corelation 
                if i == 0:
                    
                    sig = np.concatenate((SmoothRadius,SmoothRadius)) - SmoothRadius.mean()
                    lag = 0
                    
                else:
                    sig2 = np.concatenate((SmoothRadius,SmoothRadius)) - SmoothRadius.mean()
                    corr = correlate(sig2, sig)
                    lags = correlation_lags(len(sig), len(sig2))
                    
                    sig = sig2
                    
                    #FirstRadius = SmoothRadius
                    
                    lag += lags[np.argmax(corr)]
                    
                    SmoothRadius = np.roll(SmoothRadius, -lag)
                                    

                SmoothX,SmoothY = vf.ToCart(Alphanew, SmoothRadius, angle='deg')
                
                
                if ShapePlot:
                    plt.figure(fig1.number)
                    plt.plot(SmoothX*Scale,SmoothY*Scale,'-', ms = 0.1, lw = 0.5)

                    plt.figure(fig2.number)
                    plt.plot(Alphanew,SmoothRadius*Scale,'-', ms = 0.1, lw = 0.5)
                    

            if ShapePlot:
                plt.show()
                fig1.savefig(P + '\\Figures\\' + s + '_Shape.png')
                fig2.savefig(P + '\\Figures\\' + s + '_Radius.png')  
            
            # Storing contour data
            data = {'Time (min)':Ti*np.ones(len(SmoothX)), 
                        'Img':i*np.ones(len(SmoothX)),
                        'X':SmoothX,
                        'Y':SmoothY,  
                        'R':SmoothRadius,  
                        'Alpha':Alphanew} 
            
            currentData = pd.DataFrame(data=data,index = np.repeat(s,len(SmoothX))) 
            ContourData = ContourData.append(currentData) # adding to global dataframe
            
    return(ContourData,sizeData)

## 3. Enter data and run

### 3.1 Data details

In [5]:
Scale1 = 2.5 # Spatial scale (px/µm) 16-02-2021

Scale2 = 0.44 # Spatial scale (px/µm) 18-03-2021

Scale3 = 0.58 # Spatial scale (px/µm) 19-03-2021


# Paths to data
P1 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\Manips\2021.02.16'

P2 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\Manips\2021.03.18'

P3 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\Manips\2021.03.19'


# List of stacks to analyse

# experiment of 16-02-2021
# outside traps
StackListFree210216 = ['Propagule-03','Propagule-09','Propagule-12','Propagule-18','Propagule-19',
                'Propagule-24','Propagule-25']

# inside traps
StackListTrapped210216 = ['Propagule-04','Propagule-06','Propagule-07','Propagule-08',
                    'Propagule-13','Propagule-14','Propagule-15','Propagule-16','Propagule-17',
                    'Propagule-20','Propagule-21','Propagule-22','Propagule-23','Propagule-26',
                    'Propagule-27','Propagule-28','Propagule-29','Propagule-30']


# experiments of 18-03-2021
StackList210318 = ['PPG1','PPG2','PPG3','PPG4','PPG5','PPG6','PPG7','PPG8','PPG9','PPG10','PPG11',
                   'PPG12','PPG13','PPG14','PPG15','PPG16','PPG17','PPG18','PPG19','PPG20','PPG21',
                   'PPG22','PPG23','PPG24','PPG25','PPG26','PPG27','PPG28','PPG29']
            # No Rhizoid
StackList210318 = ['PPG2','PPG3','PPG5','PPG6','PPG7','PPG8','PPG9','PPG10','PPG11',
                   'PPG14','PPG15','PPG16','PPG18','PPG19',
                   'PPG22','PPG23','PPG24','PPG25','PPG26','PPG27','PPG28','PPG29']


# experiments of 19-03-2021 (68h, cropped when air arrives)
StackList210319 = ['PPG1','PPG2','PPG5','PPG6','PPG8','PPG9','PPG10','PPG11',
                   'PPG12','PPG13','PPG14','PPG15','PPG16','PPG17','PPG18','PPG19','PPG20','PPG21',
                   'PPG22','PPG23']



# CSV file names with time, size and focus data
filename210216Free = 'DataFreePpgles210216.csv'
filename210216Trapped = 'DataTrappedPpgles210216.csv'
filename210318 = 'Data210318.csv'
filename210319 = 'Data210319.csv'


### 3.2 Run

In [6]:
# Contour computation

ContourData210318,sizeData210318 = GetContours(StackList210318,P2,filename210318, Scale2, ShapePlot = False)

ContourData210319,sizeData210319 = GetContours(StackList210319,P3,filename210319, Scale3, ShapePlot = False)



In [7]:
# Saving

ContourData210318.to_csv(P2 + '\\ContourData210318.csv',index_label = 'Ind')

ContourData210319.to_csv(P3 + '\\ContourData210319.csv',index_label = 'Ind')

In [ ]:
# Computig average contour for each time point

MeanContour(StackList210318,'ContourData210318.csv',P2,'180321')
MeanContour(StackList210319,'ContourData210319.csv',P3,'190321')

# Test Zone

In [ ]:


StackList = StackList210319
datafilename = 'ContourData210319.csv'
ContourData = pd.read_csv(P3 + '\\' + datafilename, index_col = 'Ind')
ContourData['alignR'] = np.nan

refR = ContourData[ContourData['Img'] == 0][['R']].loc[StackList[0]].to_numpy()
sig = np.concatenate((refR,refR)) - refR.mean()

s = StackList[1]

currentR = ContourData[ContourData['Img'] == 0][['R']].loc[s].to_numpy()

sig2 = np.concatenate((currentR,currentR)) - currentR.mean()

# Aligining with reference contour using cross corelation 
corr = correlate(sig2, sig)

lags = correlation_lags(len(sig), len(sig2))

corr /= np.max(corr)

lag = lags[np.argmax(corr)]

alignR = np.roll(currentR, -lag)

print(alignR[:,0].shape)

ContourData.loc[(ContourData.index == s) & (ContourData['Img'] == 0),'alignR'] = alignR[:,0]





